This notebook investigates the loss trends for a GNN model training,
and looks at the worst performing scenarios.

In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import display, Markdown

import sim_ranking as sr
import ml_tools as mlt

%matplotlib inline

In [ ]:
wdata = "/Users/claudy/dev/work/data"
results_dir = "/Users/claudy/dev/work/data/sim_ranking/results/gnn/0303_1332_cv_v4p2NZGMDB_v2p1"

In [ ]:
# Injected Parameters
wdata = "/Users/claudy/dev/work/data"
results_dir = "/Users/claudy/dev/work/data/sim_ranking/results/gnn/archive/0227_1506_cv_v4p2NZGMDB_v2p3_pSAOnly"


In [ ]:
wdata = Path(wdata)
results_dir = Path(results_dir)
cim_results_dir = results_dir / "cim_results"
warnings.simplefilter(action='ignore', category=FutureWarning)

**Result Directory:** `{python} str(results_dir)`

In [ ]:
# Load metrics
metrics = pd.read_pickle(results_dir / "metrics.pickle")

cv_dirs = [cur_result for cur_result in results_dir.glob("cv_*") if cur_result.is_dir()]
cv_metrics = {cur_cv_dir.name: pd.read_parquet(cur_cv_dir / "metrics.parquet") for cur_cv_dir in cv_dirs}
cv_agg_metrics = {cur_cv_dir.name: pd.read_pickle(cur_cv_dir / "agg_metrics.pickle") for cur_cv_dir in cv_dirs}

In [ ]:
# Load observed data 
run_config = sr.ml.gnn_gm.RunConfig.from_yaml(results_dir / "run_config.yaml")
nzgmdb_ffp = wdata / run_config.rel_obs_data_ffp
obs_data = sr.data.load_obs_nzgmdb(nzgmdb_ffp)

In [ ]:
# Load the combined validation results
val_results = pd.read_parquet(results_dir / "val_results.parquet")

cim_val_results = None
if (cim_results_dir / "val_results.parquet").exists():
    cim_val_results = pd.read_parquet(cim_results_dir / "val_results.parquet").sort_index()

In [ ]:
# Compute residuals
val_res_df = sr.ml.gnn_gm.get_residuals(val_results, ims=run_config.ims)
assert val_res_df.index.equals(val_results.index)
val_res_df["cv_iter"] = val_results["cv_iter"]

In [ ]:
# Distance matrix
dist_matrix = sr.utils.calculate_distance_matrix(obs_data.sites, obs_data.site_df)

In [ ]:
# Load the empirical GMM predictions
emp_gmm_params = sr.data.load_emp_gmm_params( wdata / "sim_ranking/emp_gm_params/nzgmdb_v4p2" / "emp_gm_params_400MaxRjb.parquet" )

In [ ]:
# Compute mean weighted loss (across IMs)
im_wloss_cols = mlt.array_utils.numpy_str_join("_", run_config.ims, "w_loss")

pSA_im_wloss_cols = [f"{col}_w_loss" for col in sr.constants.PSA_KEYS]
non_pSA_im_wloss_cols = [f"{col}_w_loss" for col in sr.constants.NON_PSA_IMs]

val_results["mean_w_loss"] = val_results[pSA_im_wloss_cols].mean(axis=1)
val_results["max_w_loss"] = val_results[pSA_im_wloss_cols].max(axis=1)

## Metric Comparison Between CV-Iterations

In [ ]:
metric_keys = ["w_loss_hist", "loss_hist", "mse_hist", "mean_sigma_hist"]

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(metric_keys) * 2, 2, -1, (8, 6))


for ix, cur_metric_key in enumerate(metric_keys):
	ax1, ax2 = axs[ix * 2], axs[ix * 2 + 1]
    
    # Get current metric values
	cur_train_metric_df = metrics.sel[:, :, cur_metric_key + "_train"]
	cur_val_metric_df = metrics.sel[:, :, cur_metric_key + "_val"]
    
    # Compute mean and std
	cur_mean = cur_train_metric_df.mean(axis=1)
	cur_std = cur_train_metric_df.std(axis=1)
	
	cur_val_mean = cur_val_metric_df.mean(axis=1)
	cur_val_std = cur_val_metric_df.std(axis=1)
    
    # Compute y-axis limits
	y_center = (cur_mean.iloc[-10:].mean() + cur_val_mean.iloc[-10:].mean()) / 2
	y_width = max(cur_std.iloc[-10:].mean(), cur_val_std.iloc[-10:].mean()) * 3
	y_min = y_center - y_width
	y_max = y_center + y_width
    
    # Plot
	ax1.plot(cur_train_metric_df.index.values, cur_train_metric_df, c="gray", linewidth=1.0, linestyle="--")
	ax1.plot(cur_mean.index.values, cur_mean, c="black", linewidth=2.0)
	ax1.set_ylim(y_min, y_max)
	ax1.set_title(f"Training {cur_metric_key}")
	ax1.set_xlim(cur_train_metric_df.index.values.min(), cur_train_metric_df.index.values.max())
	ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    
	ax2.plot(cur_val_metric_df.index.values, cur_val_metric_df, c="gray", linewidth=1.0, linestyle="--")
	ax2.plot(cur_val_mean.index.values, cur_val_mean, c="black", linewidth=2.0)
	ax2.set_ylim(y_min, y_max)
	ax2.set_title(f"Validation {cur_metric_key}")
	ax2.set_xlim(cur_train_metric_df.index.values.min(), cur_train_metric_df.index.values.max())
	ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    
fig.tight_layout()

## Loss

### IM Loss Histograms

In [ ]:
plot_ims = ["pSA_0.01", "pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_3.0", "pSA_10.0"]

if run_config.im_set == sr.constants.IMSet.all:
    plot_ims.extend(sr.constants.NON_PSA_IMs)

In [ ]:
n_bins = 50

fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), 2, -1, (8, 6))

for cur_ax, cur_im  in zip(axs, plot_ims):

    sns.histplot(val_results, x=f"{cur_im}_w_loss", bins=n_bins, ax=cur_ax, common_bins=True, hue="cv_iter", multiple="stack", legend=False, palette="viridis")

    # cur_ax.hist(val_results[f"{cur_im}_w_loss"], bins=50)
    cur_ax.set_title(sr.utils.get_nice_im_name(cur_im))
    cur_ax.set_xlabel("Weighted Loss")
    cur_ax.set_ylabel("Count")
    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")

fig.tight_layout()

### Mean IM Loss

In [ ]:
# Average IM loss
mean_im_loss = val_results[im_wloss_cols].mean(axis=0)
std_im_loss = val_results[im_wloss_cols].std(axis=0)
cv_keys = val_results["cv_iter"].unique()

fig = plt.figure(figsize=(16, 6))

main_grid = gridspec.GridSpec(1, 2, figure=fig, wspace=0.03, width_ratios=[5, 1])

ax1 = fig.add_subplot(main_grid[0])
ax2 = fig.add_subplot(main_grid[1])

for cur_cv_key in cv_keys:
    ax1.plot(sr.constants.PERIODS, val_results.loc[val_results["cv_iter"] == cur_cv_key, pSA_im_wloss_cols].mean(axis=0), c="gray", alpha=0.5, linestyle="--", linewidth=1.0) 

ax1.plot(sr.constants.PERIODS, mean_im_loss[pSA_im_wloss_cols], c="b")
ax1.fill_between(sr.constants.PERIODS, mean_im_loss[pSA_im_wloss_cols] - std_im_loss[pSA_im_wloss_cols], mean_im_loss[pSA_im_wloss_cols] + std_im_loss[pSA_im_wloss_cols], alpha=0.2, color="blue")

ax1.set_xlim(0.01, 10)
ax1.set_xscale("log")
ax1.grid(which="both", linewidth=0.5, alpha=0.5, linestyle="--")
ax1.set_xlabel("Period (s)")
ax1.set_ylabel("Weighted Loss")


if run_config.im_set == sr.constants.IMSet.all:
    for cur_cv_key in cv_keys:
        ax2.scatter(sr.constants.NON_PSA_IMs, val_results.loc[val_results["cv_iter"] == cur_cv_key, non_pSA_im_wloss_cols].mean(axis=0), c="gray", alpha=0.5, marker="o", s=10)


    ax2.errorbar(sr.constants.NON_PSA_IMs, mean_im_loss[non_pSA_im_wloss_cols], yerr=std_im_loss[non_pSA_im_wloss_cols], fmt="o", c="b")

ax2.set_yticklabels([])
ax2.set_ylim(ax1.get_ylim())
ax2.grid(which="both", linewidth=0.5, alpha=0.5, linestyle="--")
ax2.xaxis.set_tick_params(rotation=90)


### CV-Based Loss

In [ ]:
mean_cv_wloss = val_results.groupby("cv_iter")["mean_w_loss"].mean()
std_cv_wloss = val_results.groupby("cv_iter")["mean_w_loss"].std()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

# ax.scatter(np.arange(len(mean_cv_wloss)), mean_cv_wloss, c="gray", alpha=0.5, marker="o", s=10)
ax.errorbar(np.arange(len(mean_cv_wloss)), mean_cv_wloss, std_cv_wloss, c="gray", alpha=0.5)

ax.axhline(mean_cv_wloss.mean(), c="b")
ax.fill_between(np.arange(len(mean_cv_wloss)), mean_cv_wloss.mean() - std_cv_wloss.mean(), mean_cv_wloss.mean() + std_cv_wloss.mean(), alpha=0.2, color="blue")

ax.set_xlabel("CV Iteration")
ax.set_ylabel("Weighted Loss")
ax.set_xlim(0, len(mean_cv_wloss))
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")

fig.tight_layout()

### Event-Based Loss

In [ ]:
mean_event_wloss = val_results.groupby("event_id")["mean_w_loss"].mean()
std_event_wloss = val_results.groupby("event_id")["mean_w_loss"].std()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.scatter(np.arange(len(mean_event_wloss)), mean_event_wloss, c="gray", alpha=0.5, marker="o", s=10)
# ax.errorbar(np.arange(len(mean_event_wloss)), mean_event_wloss, std_event_wloss, c="gray", alpha=0.5)

ax.axhline(mean_event_wloss.mean(), c="b")
ax.fill_between(np.arange(len(mean_event_wloss)), mean_event_wloss.mean() - std_event_wloss.mean(), mean_event_wloss.mean() + std_event_wloss.mean(), alpha=0.2, color="blue")

ax.set_xlabel("Event")
ax.set_ylabel("Weighted Loss")
ax.set_xlim(0, len(mean_event_wloss))
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")

fig.tight_layout()

## Site-Based Loss

In [ ]:
mean_site_wloss = val_results.groupby("site_int")["mean_w_loss"].mean()
std_site_wloss = val_results.groupby("site_int")["mean_w_loss"].std()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.scatter(np.arange(len(mean_site_wloss)), mean_site_wloss, c="gray", alpha=0.5, marker="o", s=10)

ax.axhline(mean_site_wloss.mean(), c="b")
ax.fill_between(np.arange(len(mean_site_wloss)), mean_site_wloss.mean() - std_site_wloss.mean(), mean_site_wloss.mean() + std_site_wloss.mean(), alpha=0.2, color="blue")

ax.set_xlabel("Site")
ax.set_ylabel("Weighted Loss")
ax.set_xlim(0, len(mean_site_wloss))
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")

fig.tight_layout()

## $R_{Rup}$ Loss

In [ ]:
# bin_centers, bin_means, bin_stds = mlt.utils.compute_binned_trend(obs_data.record_df.loc[val_results.index, "rrup"], val_results["mean_w_loss"], n_bins=25)
bin_centers, bin_means, bin_stds = mlt.utils.compute_count_binned_trend(obs_data.record_df.loc[val_results.index, "rrup"], val_results["mean_w_loss"], n_points_per_bin=1000)

fig, ax = plt.subplots(figsize=(16, 6))

ax.scatter(obs_data.record_df.loc[val_results.index, "rrup"], val_results["mean_w_loss"], c="gray", alpha=0.5, marker="o", s=10)
ax.plot(bin_centers, bin_means, c="b")
ax.fill_between(bin_centers, bin_means - bin_stds, bin_means + bin_stds, alpha=0.2, color="blue")

ax.set_xlabel("Rrup (km)")
ax.set_ylabel("Weighted Loss")
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax.set_ylim(None, 10)

fig.tight_layout()

## Magnitude Loss

In [ ]:
# bin_centers, bin_means, bin_stds = mlt.utils.compute_binned_trend(obs_data.record_df.loc[val_results.index, "mag"], val_results["mean_w_loss"], n_bins=25)
bin_centers, bin_means, bin_stds = mlt.utils.compute_count_binned_trend(obs_data.record_df.loc[val_results.index, "mag"], val_results["mean_w_loss"], n_points_per_bin=1000)

fig, ax = plt.subplots(figsize=(16, 6))

ax.scatter(obs_data.record_df.loc[val_results.index, "mag"], val_results["mean_w_loss"], c="gray", alpha=0.5, marker="o", s=10)
ax.plot(bin_centers, bin_means, c="b")
ax.fill_between(bin_centers, bin_means - bin_stds, bin_means + bin_stds, alpha=0.2, color="blue")

ax.set_xlabel("Magnitude (mag)")
ax.set_ylabel("Weighted Loss")
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
# ax.set_ylim(None, 10)

fig.tight_layout()

## CV- Worst Loss
Investigate the extreme loss, by looking at the worst X scenarios

In [ ]:
ext_val_results = val_results.copy(deep=True)
worst_keys = ext_val_results.sort_values("mean_w_loss")[-1000:].index.values.astype(str)
ext_val_results = ext_val_results.loc[worst_keys]

# worst_keys = []
# for cur_cv_key in cv_keys:
#     cur_cv_mask = ext_val_results["cv_iter"] == cur_cv_key
#     cur_worst_keys = ext_val_results.loc[cur_cv_mask, im_wloss_cols].mean(axis=1).sort_values().index[-100:].values.astype(str)
#     worst_keys.extend(cur_worst_keys)

# assert np.unique(worst_keys).size == len(worst_keys)
# ext_val_results = ext_val_results.loc[worst_keys]

In [ ]:
n_bins = 50

fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), 2, -1, (8, 6))

for cur_ax, cur_im  in zip(axs, plot_ims):

    sns.histplot(ext_val_results, x=f"{cur_im}_w_loss", bins=n_bins, ax=cur_ax, common_bins=True, hue="cv_iter", multiple="stack", legend=False, palette="viridis")

    # cur_ax.hist(val_results[f"{cur_im}_w_loss"], bins=50)
    cur_ax.set_title(sr.utils.get_nice_im_name(cur_im))
    cur_ax.set_xlabel("Weighted Loss")
    cur_ax.set_ylabel("Count")
    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")

fig.tight_layout()

In [ ]:
# Average IM loss
mean_im_loss = ext_val_results[im_wloss_cols].mean(axis=0)
std_im_loss = ext_val_results[im_wloss_cols].std(axis=0)
cv_keys = ext_val_results["cv_iter"].unique()

fig = plt.figure(figsize=(16, 6))

main_grid = gridspec.GridSpec(1, 2, figure=fig, wspace=0.03, width_ratios=[5, 1])

ax1 = fig.add_subplot(main_grid[0])
ax2 = fig.add_subplot(main_grid[1])

for cur_cv_key in cv_keys:
    ax1.plot(sr.constants.PERIODS, ext_val_results.loc[ext_val_results["cv_iter"] == cur_cv_key, pSA_im_wloss_cols].mean(axis=0), c="gray", alpha=0.5, linestyle="--", linewidth=1.0) 

ax1.plot(sr.constants.PERIODS, mean_im_loss[pSA_im_wloss_cols], c="b")
ax1.fill_between(sr.constants.PERIODS, mean_im_loss[pSA_im_wloss_cols] - std_im_loss[pSA_im_wloss_cols], mean_im_loss[pSA_im_wloss_cols] + std_im_loss[pSA_im_wloss_cols], alpha=0.2, color="blue")

ax1.set_xlim(0.01, 10)
ax1.set_xscale("log")
ax1.grid(which="both", linewidth=0.5, alpha=0.5, linestyle="--")
ax1.set_xlabel("Period (s)")
ax1.set_ylabel("Weighted Loss")


if run_config.im_set == sr.constants.IMSet.all:
    for cur_cv_key in cv_keys:
        ax2.scatter(sr.constants.NON_PSA_IMs, ext_val_results.loc[ext_val_results["cv_iter"] == cur_cv_key, non_pSA_im_wloss_cols].mean(axis=0), c="gray", alpha=0.5, marker="o", s=10)


    ax2.errorbar(sr.constants.NON_PSA_IMs, mean_im_loss[non_pSA_im_wloss_cols], yerr=std_im_loss[non_pSA_im_wloss_cols], fmt="o", c="b")

ax2.set_yticklabels([])
ax2.set_ylim(ax1.get_ylim())
ax2.grid(which="both", linewidth=0.5, alpha=0.5, linestyle="--")
ax2.xaxis.set_tick_params(rotation=90)


### CV-Based Loss

In [ ]:
ext_mean_cv_wloss = ext_val_results.groupby("cv_iter")["mean_w_loss"].mean()
ext_std_cv_wloss = ext_val_results.groupby("cv_iter")["mean_w_loss"].std()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

# ax.scatter(np.arange(len(ext_mean_cv_wloss)), ext_mean_cv_wloss, c="gray", alpha=0.5, marker="o", s=10)
ax.errorbar(np.arange(len(ext_mean_cv_wloss)), ext_mean_cv_wloss, ext_std_cv_wloss, c="gray", alpha=0.5)

ax.axhline(ext_mean_cv_wloss.mean(), c="b")
ax.fill_between(np.arange(len(ext_mean_cv_wloss)), ext_mean_cv_wloss.mean() - ext_std_cv_wloss.mean(), ext_mean_cv_wloss.mean() + ext_std_cv_wloss.mean(), alpha=0.2, color="blue")

ax.set_xlabel("CV Iteration")
ax.set_ylabel("Weighted Loss (Worst Cases)")
ax.set_xlim(0, len(ext_mean_cv_wloss))
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax.set_title("Cross-Validation Mean Weighted Loss for Worst Scenarios")

fig.tight_layout()

In [ ]:
ext_cv_groups = ext_val_results.groupby("cv_iter")
ext_cv_count = ext_cv_groups.size().sort_values()

fig, ax =  plt.subplots(figsize=(16, 6))

ax.bar(ext_cv_count.index, ext_cv_count.values, color="gray", alpha=0.5)

ax.xaxis.set_tick_params(rotation=90)

ax.set_xlabel("CV Iteration")
ax.set_ylabel("Count")

fig.tight_layout()

### Event-Based Loss

In [ ]:
ext_mean_event_wloss = ext_val_results.groupby("event_id")["mean_w_loss"].mean()
ext_std_event_wloss = ext_val_results.groupby("event_id")["mean_w_loss"].std()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.scatter(np.arange(len(ext_mean_event_wloss)), ext_mean_event_wloss, c="gray", alpha=0.5, marker="o", s=10)
# ax.errorbar(np.arange(len(ext_mean_event_wloss)), ext_mean_event_wloss, ext_std_event_wloss, c="gray", alpha=0.5)

ax.axhline(ext_mean_event_wloss.mean(), c="b")
ax.fill_between(np.arange(len(ext_mean_event_wloss)), ext_mean_event_wloss.mean() - ext_std_event_wloss.mean(), ext_mean_event_wloss.mean() + ext_std_event_wloss.mean(), alpha=0.2, color="blue")

ax.set_xlabel("Event")
ax.set_ylabel("Weighted Loss")
ax.set_xlim(0, len(ext_mean_event_wloss))
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")

fig.tight_layout()

In [ ]:
ext_event_count = ext_val_results.groupby("event_id").size().sort_values()

fig, ax =  plt.subplots(figsize=(16, 6))
ax.bar(np.arange(ext_event_count.shape[0]), ext_event_count.values, color="gray", alpha=0.5)
ax.set_xlabel("Event")
ax.set_ylabel("Count")
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")

fig.tight_layout()

In [ ]:
print(f"Worst 5 Events by Count")
for cur_event in ext_event_count.index[-5:]:
    print(f"Event {cur_event} - Magnitude: {obs_data.event_df.loc[cur_event, 'mag']}, Count: {ext_event_count[cur_event]}")

## Site-Based Loss

In [ ]:
ext_mean_site_wloss = ext_val_results.groupby("site_int")["mean_w_loss"].mean()
ext_std_site_wloss = ext_val_results.groupby("site_int")["mean_w_loss"].std()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.scatter(np.arange(len(ext_mean_site_wloss)), ext_mean_site_wloss, c="gray", alpha=0.5, marker="o", s=10)

ax.axhline(ext_mean_site_wloss.mean(), c="b")
ax.fill_between(np.arange(len(ext_mean_site_wloss)), ext_mean_site_wloss.mean() - ext_std_site_wloss.mean(), ext_mean_site_wloss.mean() + ext_std_site_wloss.mean(), alpha=0.2, color="blue")

ax.set_xlabel("Site")
ax.set_ylabel("Weighted Loss")
ax.set_xlim(0, len(ext_mean_site_wloss))
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")

fig.tight_layout()

In [ ]:
ext_site_count = ext_val_results.groupby("site_int").size().sort_values()

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(np.arange(ext_site_count.shape[0]), ext_site_count.values, color="gray", alpha=0.5)
ax.set_xlabel("Site")
ax.set_ylabel("Count")
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")

fig.tight_layout()

In [ ]:
print("Worst 5 Sites by Count")
for cur_site in ext_site_count.index[-5:]:
    print(f"Site {cur_site} - Count: {ext_site_count[cur_site]}")

## $R_{Rup}$ Loss

In [ ]:
# bin_centers, bin_means, bin_stds = mlt.utils.compute_binned_trend(obs_data.record_df.loc[ext_val_results.index, "rrup"], ext_val_results["mean_w_loss"], n_bins=25)
bin_centers, bin_means, bin_stds = mlt.utils.compute_count_binned_trend(obs_data.record_df.loc[ext_val_results.index, "rrup"], ext_val_results["mean_w_loss"], n_points_per_bin=50)

fig, ax = plt.subplots(figsize=(16, 6))

ax.scatter(obs_data.record_df.loc[ext_val_results.index, "rrup"], ext_val_results["mean_w_loss"], c="gray", alpha=0.5, marker="o", s=10)
ax.plot(bin_centers, bin_means, c="b")
ax.fill_between(bin_centers, bin_means - bin_stds, bin_means + bin_stds, alpha=0.2, color="blue")

ax.set_xlabel("Rrup (km)")
ax.set_ylabel("Weighted Loss")
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
# ax.set_ylim(None, 10)

fig.tight_layout()

## Magnitude Loss


In [ ]:
# bin_centers, bin_means, bin_stds = mlt.utils.compute_binned_trend(obs_data.record_df.loc[ext_val_results.index, "mag"], ext_val_results["mean_w_loss"], n_bins=25)
bin_centers, bin_means, bin_stds = mlt.utils.compute_count_binned_trend(obs_data.record_df.loc[ext_val_results.index, "mag"], ext_val_results["mean_w_loss"], n_points_per_bin=50)

fig, ax = plt.subplots(figsize=(16, 6))

ax.scatter(obs_data.record_df.loc[ext_val_results.index, "mag"], ext_val_results["mean_w_loss"], c="gray", alpha=0.5, marker="o", s=10)
ax.plot(bin_centers, bin_means, c="b")
ax.fill_between(bin_centers, bin_means - bin_stds, bin_means + bin_stds, alpha=0.2, color="blue")

ax.set_xlabel("Magnitude (mag)")
ax.set_ylabel("Weighted Loss")
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
# ax.set_ylim(None, 10)

fig.tight_layout()

## Worst Scenarios

In [ ]:
def display_scenario(cur_event: str, cur_site: str):
    display(Markdown(f"### Event: {cur_event}, SoI: {cur_site}"))
    cur_id = f"{cur_event}_{cur_site}"
    if cur_id in val_results.index:
        print(f"Magnitude: {obs_data.event_df.loc[cur_event].mag}, Rrup: {obs_data.record_df.loc[cur_id, 'rrup']:.2f}, Mean Weighted Loss: {val_results.loc[cur_id, 'mean_w_loss']:.2f}")
        # Plots
        cur_obs_sites = sr.plot_ind_scenarios.get_obs_sites(cur_event, cur_site, val_results, dist_matrix, n_obs_sites=n_obs_sites)
        # create_4plot(cur_event, cur_site, cur_obs_sites, gnn_val_results, cim_val_results, obs_data, dist_matrix)
        fig = sr.plot_ind_scenarios.create_2plot_log(cur_event, cur_site, cur_obs_sites, val_results, cim_val_results, obs_data, dist_matrix, emp_gmm_params=emp_gmm_params)
        plt.show(fig)
        plt.close(fig)


        # Info
        site_info_df = sr.plot_ind_scenarios.get_site_info_df(cur_event, cur_site, cur_obs_sites, obs_data, dist_matrix)
        display(site_info_df)
        display(val_results.loc[f"{cur_event}_{cur_site}", im_wloss_cols].to_frame().T)
    else:
        print(f"No results for {cur_event}_{cur_site}")

In [ ]:
top = 100
n_obs_sites = 10

### Worst Mean Loss

In [ ]:
worst_mean_sc_keys = val_results["mean_w_loss"].sort_values().iloc[-top:].index.astype(str)

In [ ]:
for cur_key in worst_mean_sc_keys[::-1]:
    cur_event, cur_site = cur_key.split("_")
    display_scenario(cur_event, cur_site)

In [ ]:
### Worst Max Loss

In [ ]:
# worst_max_sc_keys = val_results["max_w_loss"].sort_values().iloc[-top:].index.astype(str)

In [ ]:
# for cur_key in worst_max_sc_keys:
#     cur_event, cur_site = cur_key.split("_")
#     display_scenario(cur_event, cur_site)